### **Table of Content**
- Chapter 1.0: Introduction to Prompt Engineering

### **Key Highlights**
-
-
-

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv  # Import the specific functions from dotenv
from openai import OpenAI, RateLimitError  # Import the specific error & OpenAI

#### **1.0 Introduction to Prompt Engineering**

Prompt engineering refers to crafting effective prompts to guide the language model towards the intended response. By refining your prompts, you can achieve better results and guide the model towards generating more accurate and useful responses. 

In [2]:
# find_dotenv() will automatically climb up from 'developer_path' 
# to the root folder to find your .env file flawlessly.
load_dotenv(find_dotenv())

# Retrieve the key
api_key = os.getenv("OPENAI_API_KEY")

# Safety check to make sure it loaded
if not api_key:
    raise ValueError("API Key is still missing! Double-check the variable name inside your .env file.")

# Initialize the client
client = OpenAI(api_key=api_key)
print("Connected successfully! OpenAI client is ready.")

Connected successfully! OpenAI client is ready.


In [3]:
# Creating a function for the prompt
def get_response(prompt): # param to pass user input/instructions
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=100,
        messages=[
            {"role": "system", "content": "You are a helpful general assistant."}, # or create var prompt to insert user input/instructions
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content.strip()

#### **2.0 Key Principles of Prompt Engineering**
- Suitable verbs
- Clear & precise prompts
- Well structured delimited prompts


In [ ]:
prompt = "What is the capital of France? ```story```"
get_response(prompt)

print("")

#### **3.0 Structured Outputs & Conditional Prompts**


In [ ]:
# Table formatting
prompt = "Generate a table containing 5 rows and 3 columns with random data."
# List formatting
prompt = "Generate a list of 5 random items."
# Structured data formatting
prompt = "Generate a JSON object with 3 key-value pairs representing a person's name, age, and occupation."

# Custom output
text = "Once a upon a time in a land far, far away [...]"
instructions = "You will be provided a text delimited by triple backticks. Your task is to summarize the text in one sentence."
output_format = """
Provide the summary in the following format: 
- Text: <the original text>
- Title: <a title for the text>
- Summary: <the one-sentence summary>
"""
prompt = instructions + output_format + f"```{text}```"

# Conditional prompting
text = ""
prompt = f"""
You will be provided with a text delimited by triple backticks. If the text is written in English,
suugest title for it. Otherwise, respond with "The text is not in English."

```{text}```
"""


response = get_response(prompt)
print(response)


#### **5.0 Few-shot Prompting**
- Model provided with examples (question-answer pair)
- Number of examples: zero (zero-shot prompting), one (one-shot prompting), more than one (few shot prompting)


In [ ]:
# zero-shot prompting
prompt = "What is prompt engineering?"

# one-shot prompting
prompt = "What is prompt engineering? Here's an example of a good answer: Prompt engineering is the process of designing and refining prompts to effectively communicate with AI models, ensuring they understand the task and provide accurate responses."

# few-shot prompting
prompt = """
Text: I love programming in Python. -> Classsification: Positive
Text: I dislike bugs in my code. -> Classification: Negative
Text: I like to play video games. -> Classification: Positive
Text: I find debugging frustrating. -> Classification: 
"""

In [ ]:
# Creating the chat completion with the messages array to include the system, user, and assistant roles
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"user", 
            "content":"I love programming in Python." 
        },

        {
            "role":"assistant", 
            "content":"positive"
        },

        {
            "role":"use", 
            "content":"I dislike bugs in my code."
        },

        {
            "role":"user", 
            "content":prompt
        }
    
    ],
    temperature=0.7, # controls randomness of the output
    max_tokens=100, # controls the length of the output
)

print(response.choices[0].message.content.strip())

#### **6.0 Multi-step Prompting**
- This is used for **Sequential & Cognitive tasks**
- Break down an end goal into series of steps
- Model goes through each step to give final output
- Incorporate steps inside the prompt

In [ ]:
# Single-step prompt: Writing a blog
prompt = "Compose a travel blog"

# Multi-step prompt: Writing a blog with specific sections
prompt = """Compose a travel blog with the following sections:
1. Introduction
2. Itinerary
3. Recommendations
4. Conclusion
"""

Analyzing solutions correctness of model output

In [ ]:
text = "Hello, how are you?"

# Single-step prompting
prompt = f"""
Determine if the following text is in English or not. Respond with "The text is in English." or "The text is not in English."
```{text}```
"""

# Multi-step prompting
prompt = f"""
Determine if the following text is in English or not. Respond with "The text is in English." or "The text is not in English."
Step 1: Analyze the text to identify the language.
Step 2: Based on the analysis, determine if the text is in English or not.
Text: ```{text}```
"""

#### **7.0 Chain-of-thought & Self-consistency Prompting**
- **Chain-of-thought prompting:**
    - Requires LLMs to provide reasoning steps (thoughts) before giving answer
    - Complex reasoning tasks
    - Reduce model errors


In [ ]:
std_prompt = """"
Q: You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?
A: The answer is
"""

# Step-by-step prompting
chain_prompt = """
Q: You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?
A: Let's break down the problem step by step.
"""

# Chain-of-thought prompting with few-shots
example = """
Q: You have 10 apples. You eat 2 apples and then buy 4 more apples. How many apples do you have now?
A: From 10 exisitng apples, minus 2 eaten and then add 4 more, the total number of apples is 16.
"""

question = """
Q: You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?
A: 
"""

prompt = example + question

- **Self-consistency Prompting:**
    - Generate multiple chain-of-thoughts by prompting model several times
    - Majority vote to obtain final output
    - Define by multiple prompts or prompt generating multiple responses

In [ ]:
self_consistency_instructions = """
Imagine three completely independent experts who reason differently are answering this question.
The final answer is obtained by majority vote.
The question is:
"""

problem_to_solve = "You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?"

prompt = self_consistency_instructions + problem_to_solve

#### **8.0 Iterative Prompt Engineering & Refinement**
- No prompt can be perfect initialy
- Prompt Engineering
    - Build a prompt 
    - Feed it to the model
    - Observe & analyze the output
    - Reiterate to make the prompt better

#### **9.0 Text Summarization & Expansion**
- **Text Summarization:**
    - Text summarization condense text into shorter format
    - Streamline business processes
- **Text Expansion:**
    - Generates text from ideas & bullent points
    - Improve efficiency & productivity

In [ ]:
# Google Framework for prompt design
# PERSONA: Define the role or identity the AI should assume, such as a teacher, doctor, or customer service representative.
# AIM: Clearly state the objective or goal of the prompt, specifying what you want the AI to accomplish.
# RECIPIENT: Identify the target audience or user for the AI's response, which can influence the tone and style of the output.
# THEME: Specify the context or subject matter that the AI should focus on when generating its response.
# STRUCTURE: Specify the desired format of the output, such as a list, table, or JSON object.

# Report
report = "In recent years, the rapid advancement of artificial intelligence (AI) has brought about significant benefits across various industries, including healthcare, finance, and transportation. However, this progress has also raised concerns about data privacy. AI systems often require large amounts of data to function effectively, which can include sensitive personal information. As a result, there is an increasing need for robust data privacy measures to protect individuals' information while still allowing AI to thrive. This report explores the intersection of AI and data privacy, highlighting the challenges and potential solutions in this evolving landscape."

# Craft a prompt to summarize the report
prompt = f"""
summarize the report while focusing on AI and data privacy
```{report}```
"""

response = get_response(prompt)

print("Summarized report: \n", response)



#### **10.0 Text Transformation**
- Transform given text to create a new text
- Language translation / tone adjustments / 


In [ ]:
# language translation
text = "Saya suka kecerdasan buatan."
prompt = f"""Translate the following text into English:
```{text}```
"""

# multilingual translation
prompt = f"""Translate the following text into English, French and Spanish:
```{text}```
"""

In [ ]:
# tone adjustment
text = "I am not happy with the service I received."
prompt = f"""Rewrite the following text in a more positive tone:
```{text}```
"""

# proof reading
text = "This is a sentense with a typo."
prompt = f"""Proofread the following text and correct any errors:
```{text}```
"""


In [ ]:
# grammar & writing improvement
text = "The quick brown fox jump over the lazy dog."
prompt = f"""Improve the grammar and writing style of the following text:
```{text}```
"""

# multiple transformations
text = "The quick brown fox jump over the lazy dog."
prompt = f"""Perform the following transformations on the given text:
1. Correct the grammar.
2. Improve the writing style.
```{text}```
"""

#### **11.0 Text Analysis**
- Examine text to extract information:
    - Text Classification: Sentiment Analysis
    - Entity Extraction: specific entities from text


In [ ]:
# text classification
text = "I love this product! It works great and has exceeded my expectations."
prompt = f"""
Classify the sentiment of the following text as Positive, Negative, or Neutral:
```{text}```
"""


In [ ]:
# entity extraction
text = "Apple Inc. is a technology company based in Cupertino, California."
prompt = f"""Extract the entities from the following text and classify them into categories such as Organization, Location, and Person:
```{text}```
"""

In [ ]:
# entity extraction with few shots
text_1 = "Google was founded by Larry Page and Sergey Brin while they were Ph.D. students at Stanford University."
text_2 = "Microsoft Corporation is an American multinational technology company headquartered in Redmond, Washington."
text_3 = "Amazon.com, Inc. is an American multinational technology company based in Seattle, Washington."

entities_1 = "Organization: Google, Person: Larry Page, Person: Sergey Brin"
entities_2 = "Organization: Microsoft Corporation, Location: Redmond, Location: Washington"

prompt =f"""
Text: {text_1} -> Entities: {entities_1}
Text: {text_2} -> Entities: {entities_2}
Text: {text_3} -> Entities:
"""

#### **12.0 Code Generation & Explanation**
- Code generation create source code to solve given problem


In [ ]:
# problem description
# programming language
# format (script, function, class, etc.)

# example: Write a Python function that takes a list of numbers as input and returns the average of those numbers.
prompt = "write a Python function that takes a list of numbers as input and returns the average of those numbers."

# input-output examples
examples = """
Input 1: [1, 2, 3, 4, 5] -> Output 1: 3.0
Input 2: [10, 20, 30] -> Output 2: 20.0
Input 3: [5, 10, 15] -> Output 3: 10.0
"""

prompt = f"""
You are provided with a problem description and input-output examples. Write a Python function that takes a list of numbers as input and returns the average of those numbers.
Write code that can solve the problem based on the provided examples.
```{examples}```
"""

# code modification example
script = """
quarterly_sales = [15000, 20000, 25000, 30000]
total_sales = sum(quarterly_sales)
print("Total sales for the year:", total_sales)
"""

prompt = f"""
Modify the following Python script to calculate and print the average quarterly sales instead of the total sales.
```{script}```
"""

In [ ]:
# code explanation example
script = """
def calculate_average(numbers):
    if len(numbers) == 0:
        return 0
    total = sum(numbers)    
    average = total / len(numbers)
    return average
"""

prompt = f"""
Explain the following Python script in detail.
```{script}```
"""

#### **13.0 Prompt Engineering for Chatbot Development**
- Predict user questions
- Prompt engineering to giude chatbot behavior
- Offer domain-acurate assistance

In [ ]:
# send series of messages with different roles to the API
# generic function to get response from the API with system and user prompts as parameters
def get_response(system_prompt, user_prompt): # param to pass user input/instructions
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role":"system", 
                "content":system_prompt # input from the function parameter to set the system prompt
            },

            {
                "role":"user", 
                "content":user_prompt # input from the function parameter to set the user prompt
            } # add more messages with different roles as needed
            
        ],
        temperature=0.7, # controls randomness of the output
        max_tokens=100, # controls the length of the output
    )

    return response.choices[0].message.content.strip()

In [ ]:
# fine tune the function to include assistant role and more messages as needed

base_system_prompt = ""

behavior_guidelines = ""

response_guidelines = ""

refined_prompt = base_system_prompt + behavior_guidelines + response_guidelines


def get_refined_response(refined_prompt, user_prompt): # param to pass user input/instructions
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role":"system", 
                "content":refined_prompt # input from the function parameter to set the system prompt
            },

            {
                "role":"user", 
                "content":user_prompt # input from the function parameter to set the user prompt
            } # add more messages with different roles as needed
            
        ],
        temperature=0.7, # controls randomness of the output
        max_tokens=100, # controls the length of the output
    )

    return response.choices[0].message.content.strip()

#### **14.0 Role-playing Prompts for Chatbots**
- To adopt specific role
- Tailor language & content to fit the persona
- Effective interactions


In [ ]:
# role-playing prompts
system_prompt = "You are a helpful assistant that classifies the sentiment of the given text as Positive, Negative, or Neutral."
user_prompt = "I dislike bugs in my code."

response = get_response(system_prompt, user_prompt)
print(response)

#### **15.0 Incorporating External Context**
- Use pre-trained language models recognize information they are trained on
- Lack information in LLMs: Knowledge cut-off & info requested are non-public information
- Provide more context to ensure accuracy & effectiveness: Sample of previous conversation & system prompt

In [ ]:
# sample of previous conversation with assistant role and more messages
def get_response(refined_prompt, user_prompt): # param to pass user input/instructions
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role":"system", 
                "content":refined_prompt # input from the function parameter to set the system prompt
            },

            {
                "role":"user", 
                "content":user_prompt # input from the function parameter to set the user prompt
            },

            {
                "role":"assistant", # role of the assistant to provide a response based on the system and user prompts  
                "content":"negative" # EXAMPLE response based on the user prompt and system instructions    
            }, # add more messages with different roles as needed

            {
                "role":"user", 
                "content":"" # input from the function parameter to set the user prompt
            }
            
        ],
        temperature=0.7, # controls randomness of the output
        max_tokens=100, # controls the length of the output
    )

    return response.choices[0].message.content.strip()

In [ ]:
# sample of system prompt

product_knowledge = "This ABC product is a state-of-the-art gadget designed to enhance productivity and streamline daily tasks. It features a sleek design, powerful performance, and user-friendly interface, making it an ideal choice for both professionals and casual users."

system_prompt = f"""
You are a knowledgeable assistant specializing in providing information about the ABC product. Your task is to answer user queries based on the following product knowledge:
```{product_knowledge}```
"""

def get_response(system_prompt, user_prompt): # param to pass user input/instructions
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role":"system", 
                "content":system_prompt # input from the function parameter to set the system prompt
            },

            {
                "role":"user", 
                "content":user_prompt # input from the function parameter to set the user prompt
            }
            
        ],
        temperature=0.7, # controls randomness of the output
        max_tokens=100, # controls the length of the output
    )

    return response.choices[0].message.content.strip()

#### **16.0 Example Template**


In [ ]:
# code example for a user query about the travelling

conversation = [
    {
        "role": "system",
        "content": (
            "You are a virtual Paris travel expert who provides accurate, "
            "helpful, and concise information about Paris landmarks, museums, "
            "and tourist attractions."
        )
    },
    {
        "role": "user",
        "content": "How far away is the Louvre from the Eiffel Tower (in miles) if you are driving?"
    },
    {
        "role": "assistant",
        "content": "The Louvre Museum is about 3 miles from the Eiffel Tower when driving, depending on the route and traffic conditions."
    },
    {
        "role": "user",
        "content": "Where is the Arc de Triomphe?"
    },
    {
        "role": "assistant",
        "content": "The Arc de Triomphe is located at Place Charles de Gaulle at the western end of the Champs-Élysées in Paris, France."
    },
    {
        "role": "user",
        "content": "What are the must-see artworks at the Louvre Museum?"
    },
    {
        "role": "assistant",
        "content": "Some must-see artworks at the Louvre Museum include the Mona Lisa by Leonardo da Vinci, the Venus de Milo, the Winged Victory of Samothrace, and Liberty Leading the People by Eugène Delacroix."
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=conversation,
    temperature=0.7,
    max_tokens=100,
)

print(response.choices[0].message.content.strip())



